In [4]:
import asyncio
import re
from datetime import datetime
import pandas as pd
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import requests
import nest_asyncio

# Enable nested event loop for Jupyter
nest_asyncio.apply()

# Constants
EXCEL_OUTPUT = "bodman_attorneys.xlsx"
TODAY = datetime.now().strftime("%Y-%m-%d")
COMPANY_NAME = "Bodman PLC"
HONOR_KEYWORDS = ["cum laude", "magna cum laude", "summa cum laude", "with honors", "honors"]

# Extract profile URLs from sitemap
def extract_profile_urls_from_sitemap():
    sitemap_url = "https://www.bodmanlaw.com/sitemap.xml"
    res = requests.get(sitemap_url)
    soup = BeautifulSoup(res.content, "xml")
    urls = [loc.text for loc in soup.find_all("loc") if "/people/" in loc.text]
    return urls

# Helper function to clean text
def clean(text):
    return re.sub(r'\s+', ' ', text.strip()) if text else "N/A"

# Main scraping function
async def scrape_profile(page, url):
    await page.goto(url, timeout=60000)
    await page.wait_for_load_state("load")
    html = await page.content()
    soup = BeautifulSoup(html, "html.parser")

    # Extract basic info
    name_tag = soup.select_one("div.inner > h1")
    full_name = name_tag.get_text(strip=True) if name_tag else ""
    parts = full_name.split()
    first_name = parts[0] if parts else ""
    middle_initial = parts[1].replace(".", "") if len(parts) == 3 else ""
    last_name = parts[-1] if len(parts) >= 2 else ""

    # Job title & location
    h2 = soup.select_one("div.inner > h2")
    job_title = location = "N/A"
    if h2:
        job_title = h2.contents[0].strip()
        loc_tag = h2.select_one("a")
        if loc_tag:
            location = loc_tag.get_text(strip=True)

    # Contact info
    email_tag = soup.select_one("a.email")
    email = email_tag.get("href").replace("mailto:", "") if email_tag else "N/A"
    phone_tag = soup.select_one("a.phone")
    phone = phone_tag.get_text(strip=True) if phone_tag else "N/A"

    # LinkedIn
    linkedin_tag = soup.select_one("li.linkedin a")
    linkedin = linkedin_tag.get("href") if linkedin_tag else "N/A"

    # Photo
    photo_style = soup.select_one("#printImage")
    photo_url = re.search(r'url\((.*?)\)', photo_style["style"]).group(1) if photo_style else "N/A"

    # Biography
    bio_section = soup.select_one("#overview")
    bio_parts = []
    if bio_section:
        h2 = bio_section.find("h2")
        if h2:
            bio_parts.append(clean(h2.get_text()))
        count = 0
        for tag in bio_section.find_all():
            if tag.name == "h5":
                break
            if tag.name == "p":
                bio_parts.append(clean(tag.get_text()))
                count += 1
            if count == 3:
                break
    biography = "\n\n".join(bio_parts) if bio_parts else "N/A"

    # Practice groups (updated)
    practice_section = soup.select_one("div.practices")
    practice_groups = [clean(a.get_text()) for a in practice_section.select("ul > li > a")] if practice_section else []
    practice_str = ", ".join(practice_groups) if practice_groups else "N/A"

    # Education (updated)
    law_school_name = law_school_year = law_school_honors = "N/A"
    undergrad_name = undergrad_year = undergrad_honors = "N/A"

    edu_section = soup.select_one("div.education")
    if edu_section:
        for li in edu_section.select("li"):
            text = li.get_text(" ", strip=True)
            lower = text.lower()
            year_match = re.search(r'\b(19\d{2}|20\d{2})\b', text)
            year = year_match.group() if year_match else "N/A"

            honors = next((h for h in HONOR_KEYWORDS if h in lower), "N/A")

            if "j.d." in lower or "jd" in lower:
                school = text.split(",")[0].strip()
                law_school_name = school
                law_school_year = year
                law_school_honors = honors

            elif any(deg in lower for deg in ["b.a.", "ba", "b.s.", "bs", "b.b.a.", "bba"]):
                school = text.split(",")[0].strip()
                undergrad_name = school
                undergrad_year = year
                undergrad_honors = honors

    # Bar Admissions
    admissions_section = soup.select_one("div.admissions")
    bar_admissions = [clean(li.get_text()) for li in admissions_section.select("ul > li")] if admissions_section else []
    bar_str = ", ".join(bar_admissions) if bar_admissions else "N/A"

    # Languages (NEW)
    languages_section = soup.select_one("div.languages")
    languages = [clean(li.get_text()) for li in languages_section.select("ul > li")] if languages_section else []
    languages_str = ", ".join(languages) if languages else "N/A"

    # Final dictionary
    return {
        "Company Name": COMPANY_NAME,
        "First Name": first_name,
        "Middle Initial": middle_initial,
        "Last Name": last_name,
        "Job Title": job_title,
        "Company Email": email,
        "Attorney Location(s)": location,
        "Work Phone Number": phone,
        "Biography/Overview": biography,
        "Practice Group": practice_str,
        "Specialties": "N/A",
        "Industry Focus": "N/A",
        "Law School Name": law_school_name,
        "Law School Graduation Year": law_school_year,
        "Law School Honors": law_school_honors,
        "Undergraduate School Name": undergrad_name,
        "Undergraduate Graduation Year": undergrad_year,
        "Undergraduate School Honors": undergrad_honors,
        "Bar Admissions": bar_str,
        "Languages": languages_str,
        "Attorney Website URL": url,
        "Attorney LinkedIn URL": linkedin,
        "Photo?": photo_url,
        "Date Added to Database": TODAY
    }

# Runner for Jupyter
async def main():
    profile_urls = extract_profile_urls_from_sitemap()
    all_data = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context()
        page = await context.new_page()

        for i, url in enumerate(profile_urls):
            print(f"Scraping {i+1}/{len(profile_urls)}: {url}")
            try:
                data = await scrape_profile(page, url)
                all_data.append(data)
            except Exception as e:
                print(f"❌ Error on {url}: {e}")

        await browser.close()

    # Save to Excel
    df = pd.DataFrame(all_data)
    df.to_excel(EXCEL_OUTPUT, index=False)
    print(f"\n✅ Data saved to: {EXCEL_OUTPUT}")

# For Jupyter: run with await main()
await main()


Scraping 1/203: https://www.bodmanlaw.com/people/gregory-d-degrazia/
Scraping 2/203: https://www.bodmanlaw.com/people/ann-m-mcgowan/
Scraping 3/203: https://www.bodmanlaw.com/people/james-t-ramer/
Scraping 4/203: https://www.bodmanlaw.com/people/leslie-y-vazquez/
Scraping 5/203: https://www.bodmanlaw.com/people/grace-kueber/
Scraping 6/203: https://www.bodmanlaw.com/people/arti-batra/
Scraping 7/203: https://www.bodmanlaw.com/people/heather-zimny/
Scraping 8/203: https://www.bodmanlaw.com/people/daniel-j-canine/
Scraping 9/203: https://www.bodmanlaw.com/people/jessica-d-vanwert/
Scraping 10/203: https://www.bodmanlaw.com/people/jason-m-currie/
Scraping 11/203: https://www.bodmanlaw.com/people/reem-s-aburukba/
Scraping 12/203: https://www.bodmanlaw.com/people/tatianna-a-gore/
Scraping 13/203: https://www.bodmanlaw.com/people/grace-n-heidorn/
Scraping 14/203: https://www.bodmanlaw.com/people/carson-v-garguilo/
Scraping 15/203: https://www.bodmanlaw.com/people/yasmine-h-choucair/
Scraping

In [ ]:
import asyncio
import re
from datetime import datetime
import pandas as pd
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import requests
import nest_asyncio

# Enable nested event loop for Jupyter
nest_asyncio.apply()

# Constants
EXCEL_OUTPUT = "bodman_attorneys.xlsx"
TODAY = datetime.now().strftime("%Y-%m-%d")
COMPANY_NAME = "Bodman PLC"
HONOR_KEYWORDS = ["cum laude", "magna cum laude", "summa cum laude", "with honors", "honors"]

# Extract profile URLs from sitemap
def extract_profile_urls_from_sitemap():
    sitemap_url = "https://www.bodmanlaw.com/sitemap.xml"
    res = requests.get(sitemap_url)
    soup = BeautifulSoup(res.content, "xml")
    urls = [loc.text for loc in soup.find_all("loc") if "/people/" in loc.text]
    return urls

# Helper function to clean text
def clean(text):
    return re.sub(r'\s+', ' ', text.strip()) if text else "N/A"

# Main scraping function
async def scrape_profile(page, url):
    await page.goto(url, timeout=60000)
    await page.wait_for_load_state("load")
    html = await page.content()
    soup = BeautifulSoup(html, "html.parser")

    # Extract basic info
    name_tag = soup.select_one("div.inner > h1")
    full_name = name_tag.get_text(strip=True) if name_tag else ""
    parts = full_name.split()
    first_name = parts[0] if parts else ""
    middle_initial = parts[1].replace(".", "") if len(parts) == 3 else ""
    last_name = parts[-1] if len(parts) >= 2 else ""

    # Job title & location
    h2 = soup.select_one("div.inner > h2")
    job_title = location = "N/A"
    if h2:
        job_title = h2.contents[0].strip()
        loc_tag = h2.select_one("a")
        if loc_tag:
            location = loc_tag.get_text(strip=True)

    # Contact info
    email_tag = soup.select_one("a.email")
    email = email_tag.get("href").replace("mailto:", "") if email_tag else "N/A"
    phone_tag = soup.select_one("a.phone")
    phone = phone_tag.get_text(strip=True) if phone_tag else "N/A"

    # LinkedIn
    linkedin_tag = soup.select_one("li.linkedin a")
    linkedin = linkedin_tag.get("href") if linkedin_tag else "N/A"

    # Photo
    photo_style = soup.select_one("#printImage")
    photo_url = re.search(r'url\((.*?)\)', photo_style["style"]).group(1) if photo_style else "N/A"

    # Biography
    bio_section = soup.select_one("#overview")
    bio_parts = []
    if bio_section:
        h2 = bio_section.find("h2")
        if h2:
            bio_parts.append(clean(h2.get_text()))
        count = 0
        for tag in bio_section.find_all():
            if tag.name == "h5":
                break
            if tag.name == "p":
                bio_parts.append(clean(tag.get_text()))
                count += 1
            if count == 3:
                break
    biography = "\n\n".join(bio_parts) if bio_parts else "N/A"

    # Practice groups (updated)
    practice_section = soup.select_one("div.practices")
    practice_groups = [clean(a.get_text()) for a in practice_section.select("ul > li > a")] if practice_section else []
    practice_str = ", ".join(practice_groups) if practice_groups else "N/A"

    # Education (updated with honors = degree, honors)
    law_school_name = law_school_year = law_school_honors = "N/A"
    undergrad_name = undergrad_year = undergrad_honors = "N/A"

    edu_section = soup.select_one("div.education")
    if edu_section:
        for li in edu_section.select("li"):
            text = li.get_text(" ", strip=True)
            lower = text.lower()
            year_match = re.search(r'\b(19\d{2}|20\d{2})\b', text)
            year = year_match.group() if year_match else "N/A"

            honors = next((h for h in HONOR_KEYWORDS if h in lower), "")

            if "j.d." in lower or "jd" in lower:
                school = text.split(",")[0].strip()
                law_school_name = school
                law_school_year = year
                law_school_honors = "J.D." + (", " + honors if honors else "")

            elif any(deg in lower for deg in ["b.a.", "ba", "b.s.", "bs", "b.b.a.", "bba"]):
                school = text.split(",")[0].strip()
                undergrad_name = school
                undergrad_year = year
                undergrad_honors = ("B.A." if "b.a." in lower or "ba" in lower else
                                    "B.S." if "b.s." in lower or "bs" in lower else
                                    "B.B.A." if "b.b.a." in lower or "bba" in lower else "")
                undergrad_honors = undergrad_honors + (", " + honors if honors else "")

    # Bar Admissions
    admissions_section = soup.select_one("div.admissions")
    bar_admissions = [clean(li.get_text()) for li in admissions_section.select("ul > li")] if admissions_section else []
    bar_str = ", ".join(bar_admissions) if bar_admissions else "N/A"

    # Languages (NEW)
    languages_section = soup.select_one("div.languages")
    languages = [clean(li.get_text()) for li in languages_section.select("ul > li")] if languages_section else []
    languages_str = ", ".join(languages) if languages else "N/A"

    # Final dictionary
    return {
        "Company Name": COMPANY_NAME,
        "First Name": first_name,
        "Middle Initial": middle_initial,
        "Last Name": last_name,
        "Job Title": job_title,
        "Company Email": email,
        "Attorney Location(s)": location,
        "Work Phone Number": phone,
        "Biography/Overview": biography,
        "Practice Group": practice_str,
        "Specialties": "N/A",
        "Industry Focus": "N/A",
        "Law School Name": law_school_name,
        "Law School Graduation Year": law_school_year,
        "Law School Honors": law_school_honors,
        "Undergraduate School Name": undergrad_name,
        "Undergraduate Graduation Year": undergrad_year,
        "Undergraduate School Honors": undergrad_honors,
        "Bar Admissions": bar_str,
        "Languages": languages_str,
        "Attorney Website URL": url,
        "Attorney LinkedIn URL": linkedin,
        "Photo?": photo_url,
        "Date Added to Database": TODAY
    }

# Runner for Jupyter
async def main():
    profile_urls = extract_profile_urls_from_sitemap()
    all_data = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context()
        page = await context.new_page()

        for i, url in enumerate(profile_urls):
            print(f"Scraping {i+1}/{len(profile_urls)}: {url}")
            try:
                data = await scrape_profile(page, url)
                all_data.append(data)
            except Exception as e:
                print(f"❌ Error on {url}: {e}")

        await browser.close()

    # Save to Excel
    df = pd.DataFrame(all_data)
    df.to_excel(EXCEL_OUTPUT, index=False)
    print(f"\n✅ Data saved to: {EXCEL_OUTPUT}")

# For Jupyter: run with await main()
await main()


Scraping 1/203: https://www.bodmanlaw.com/people/gregory-d-degrazia/
Scraping 2/203: https://www.bodmanlaw.com/people/ann-m-mcgowan/
Scraping 3/203: https://www.bodmanlaw.com/people/james-t-ramer/
Scraping 4/203: https://www.bodmanlaw.com/people/leslie-y-vazquez/
Scraping 5/203: https://www.bodmanlaw.com/people/grace-kueber/
Scraping 6/203: https://www.bodmanlaw.com/people/arti-batra/
Scraping 7/203: https://www.bodmanlaw.com/people/heather-zimny/
Scraping 8/203: https://www.bodmanlaw.com/people/daniel-j-canine/
Scraping 9/203: https://www.bodmanlaw.com/people/jessica-d-vanwert/
Scraping 10/203: https://www.bodmanlaw.com/people/jason-m-currie/
Scraping 11/203: https://www.bodmanlaw.com/people/reem-s-aburukba/
Scraping 12/203: https://www.bodmanlaw.com/people/tatianna-a-gore/
Scraping 13/203: https://www.bodmanlaw.com/people/grace-n-heidorn/
Scraping 14/203: https://www.bodmanlaw.com/people/carson-v-garguilo/
Scraping 15/203: https://www.bodmanlaw.com/people/yasmine-h-choucair/
Scraping